# Physis — Tahap 0: manifest, split, dan prapemrosesan GRAZPEDWRI-DX

Notebook ini menyiapkan seluruh data untuk pipeline Physis dan mengekspornya sebagai zip.

Yang dihasilkan:

- `manifest.csv` — satu baris per citra: `patient_id`, `study_id`, usia, flag exclusion, kolom `fold`, parameter geometri prapemrosesan
- `fracture_boxes.csv` — kotak fraktur yang sudah dipetakan ke ruang 384x384
- `images_384/` — seluruh 20.327 citra sudah di-resize, dipad, dan dinormalisasi
- `annot_sample/` — 100 citra normal dari fold validasi untuk anotasi lempeng pertumbuhan
- `summary.json` + tabel hitungan per band usia

**Sesi: pilih CPU, bukan GPU.** Tahap ini murni I/O dan tidak menyentuh GPU sama sekali. Memakai T4x2 hanya membakar kuota GPU mingguanmu tanpa mempercepat apa pun. Estimasi jalan: 15 sampai 25 menit.

In [1]:
import os, re, json, glob, shutil, zipfile, hashlib, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import GroupKFold

# KONFIGURASI
IMG_SIZE      = 384      # sisi panjang setelah resize, sekaligus sisi kanvas persegi
PATCH         = 16
BIT_DEPTH     = 16       # 16 = fidelitas penuh (~3 GB), 8 = separuh ukuran (~1.5 GB)
CLIP_LO, CLIP_HI = 1.0, 99.0
N_FOLDS       = 5        # fold 0 = test, fold 1 = val, fold 2..4 = train
SEED          = 1337
N_ANNOT       = 100      # citra normal fold val yang diekspor untuk anotasi manual
N_ANNOT_OVERLAP = 10     # di antaranya, yang dianotasi ketiga anggota tim
ZIP_SHARD_GB  = 1.4
KEEP_PNG_DIR  = False    # True kalau mau menyimpan hasil sebagai Kaggle Dataset, bukan zip

OUT = Path('/kaggle/working/physis_data')
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'images_384').mkdir(exist_ok=True)
(OUT / 'annot_sample').mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
print('konfigurasi siap')

konfigurasi siap


## 1. Menemukan berkas masukan

Struktur folder dataset Kaggle bisa berubah antar versi, jadi jangan hardcode path. Sel ini mencari sendiri `dataset.csv`, direktori citra, dan label YOLO.

In [4]:
ROOT = Path('/kaggle/input')
print('isi /kaggle/input:', [p.name for p in ROOT.iterdir()])

csv_candidates = sorted(ROOT.glob('**/dataset.csv'))
assert csv_candidates, 'dataset.csv tidak ditemukan'
CSV_PATH = csv_candidates[0]
BASE = CSV_PATH.parent
print('dataset.csv :', CSV_PATH)
print('base dir    :', BASE)

# indeks citra: filestem -> path, digabung lintas images_part1..4
img_index = {}
for p in BASE.glob('images_part*/**/*.png'):
    img_index[p.stem] = p
if not img_index:
    for p in BASE.glob('**/*.png'):
        if 'label' not in str(p).lower():
            img_index[p.stem] = p
print('citra ditemukan :', len(img_index))

# indeks label YOLO: filestem -> path txt
lbl_index = {}
for p in BASE.glob('**/labels/**/*.txt'):
    lbl_index[p.stem] = p
if not lbl_index:
    for p in BASE.glob('**/*.txt'):
        if p.stem in img_index:
            lbl_index[p.stem] = p
print('label ditemukan :', len(lbl_index))

yaml_hits = sorted(BASE.glob('**/*.yaml')) + sorted(BASE.glob('**/*.yml'))
print('yaml            :', [str(y.relative_to(BASE)) for y in yaml_hits][:5])

isi /kaggle/input: ['datasets']
dataset.csv : /kaggle/input/datasets/jasonroggy/grazpedwri-dx/dataset.csv
base dir    : /kaggle/input/datasets/jasonroggy/grazpedwri-dx
citra ditemukan : 20327
label ditemukan : 20327
yaml            : ['folder_structure/yolov5/meta.yaml']


## 2. Memuat `dataset.csv` dan memetakan kolomnya

17 kolom, dan namanya belum tentu persis seperti dugaan. Sel ini menormalkan nama kolom lalu mencocokkan yang dibutuhkan berdasarkan substring, dan mencetak hasil pencocokannya supaya kamu bisa memverifikasi sendiri.

In [5]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print('kolom:', list(df.columns))
print('baris:', len(df))
display(df.head(3))

def find_col(*keys, required=True):
    for k in keys:
        for c in df.columns:
            if k in c:
                return c
    if required:
        raise KeyError(f'kolom untuk {keys} tidak ketemu; cek daftar kolom di atas')
    return None

COL = {
    'stem'      : find_col('filestem', 'file_stem', 'filename'),
    'patient'   : find_col('patient_id', 'patient'),
    'age'       : find_col('age'),
    'gender'    : find_col('gender', 'sex'),
    'projection': find_col('projection', 'view', required=False),
    'laterality': find_col('laterality', 'side', required=False),
    'frac_vis'  : find_col('fracture_visible', 'fracture_v', required=False),
    'ao'        : find_col('ao_class', 'ao_'), 
    'cast'      : find_col('cast'),
    'metal_tag' : find_col('metal', required=False),
}
print()
for k, v in COL.items():
    print(f'{k:11s} -> {v}')

kolom: ['filestem', 'patient_id', 'study_number', 'timehash', 'gender', 'age', 'laterality', 'projection', 'initial_exam', 'ao_classification', 'cast', 'diagnosis_uncertain', 'osteopenia', 'fracture_visible', 'metal', 'pixel_spacing', 'device_manufacturer']
baris: 20327


,filestem,patient_id,study_number,timehash,gender,age,laterality,projection,initial_exam,ao_classification,cast,diagnosis_uncertain,osteopenia,fracture_visible,metal,pixel_spacing,device_manufacturer
0,0001_1297860395_01_WRI-L1_M014,1,1,1297860395,M,14.1,L,1,1.0,23r-M/2.1,NaN,NaN,NaN,NaN,NaN,0.144,Siemens
1,0001_1297860435_01_WRI-L2_M014,1,1,1297860435,M,14.1,L,2,1.0,23r-M/2.1,NaN,NaN,NaN,1.0,NaN,0.144,Siemens
2,0002_0354485735_01_WRI-R1_F012,2,1,354485735,F,12.0,R,1,1.0,23r-M/2.1,NaN,1.0,NaN,NaN,NaN,0.144,Siemens



stem        -> filestem
patient     -> patient_id
age         -> age
gender      -> gender
projection  -> projection
laterality  -> laterality
frac_vis    -> fracture_visible
ao          -> ao_classification
cast        -> cast
metal_tag   -> metal


## 3. Membaca label YOLO

Sembilan kelas kotak. Urutan indeksnya diambil dari `data.yaml` bila ada, kalau tidak dipakai urutan alfabetis yang dipakai rilis resminya. Sel ini memverifikasi tebakan itu dengan membandingkan jumlah kotak per kelas terhadap angka yang tertulis di kartu dataset. Kalau urutannya salah, angkanya tidak akan cocok dan assert-nya gagal.

In [6]:
DEFAULT_CLASSES = ['boneanomaly','bonelesion','foreignbody','fracture','metal',
                   'periostealreaction','pronatorsign','softtissue','text']
CLASSES = DEFAULT_CLASSES
for y in yaml_hits:
    txt = Path(y).read_text(errors='ignore')
    m = re.search(r'names\s*:\s*\[(.*?)\]', txt, re.S)
    if m:
        parsed = [s.strip().strip('\'"') for s in m.group(1).split(',')]
        if len(parsed) == 9:
            CLASSES = parsed
            print('urutan kelas dari', y.name, ':', CLASSES)
            break
else:
    print('memakai urutan default:', CLASSES)

presence = {c: defaultdict(int) for c in CLASSES}
frac_boxes_raw = defaultdict(list)   # stem -> [(xc, yc, w, h) ternormalisasi]

for stem, lp in lbl_index.items():
    try:
        lines = lp.read_text().strip().splitlines()
    except Exception:
        continue
    for ln in lines:
        parts = ln.split()
        if len(parts) < 5:
            continue
        ci = int(float(parts[0]))
        if ci >= len(CLASSES):
            continue
        cname = CLASSES[ci]
        presence[cname][stem] += 1
        if cname == 'fracture':
            xc, yc, w, h = map(float, parts[1:5])
            frac_boxes_raw[stem].append((xc, yc, w, h))

counts = {c: sum(presence[c].values()) for c in CLASSES}
print('\njumlah kotak per kelas:')
for c in CLASSES:
    print(f'  {c:20s} {counts[c]:>6d}')

EXPECTED = {'boneanomaly':276,'bonelesion':45,'foreignbody':8,'fracture':18090,
            'metal':818,'periostealreaction':3453,'pronatorsign':567,
            'softtissue':464,'text':23722}
mismatch = {c: (counts.get(c), EXPECTED[c]) for c in EXPECTED
            if abs(counts.get(c, 0) - EXPECTED[c]) > max(5, 0.02 * EXPECTED[c])}
if mismatch:
    print('\nPERINGATAN, jumlah tidak cocok dengan kartu dataset:', mismatch)
    print('kemungkinan urutan kelas berbeda; periksa sebelum lanjut')
else:
    print('\njumlah kotak cocok dengan kartu dataset, urutan kelas benar')

memakai urutan default: ['boneanomaly', 'bonelesion', 'foreignbody', 'fracture', 'metal', 'periostealreaction', 'pronatorsign', 'softtissue', 'text']

jumlah kotak per kelas:
  boneanomaly             276
  bonelesion               45
  foreignbody               8
  fracture              18090
  metal                   818
  periostealreaction     3453
  pronatorsign            567
  softtissue              464
  text                  23722

jumlah kotak cocok dengan kartu dataset, urutan kelas benar


## 4. Menurunkan `study_id`

Format nama berkas `PPPP_IIIIIIIIII_SS_...` memuat indeks pasien, id citra, dan nomor studi. Antrean radiolog beroperasi per studi, bukan per citra, jadi kolom ini dibutuhkan untuk agregasi skor. Kartu dataset menyebut 10.643 studi, dan sel ini mengecek apakah turunannya mendekati angka itu.

In [7]:
def derive_study(stem):
    parts = stem.split('_')
    if len(parts) >= 3:
        return f'{parts[0]}_{parts[2]}'
    return parts[0]

df['stem']     = df[COL['stem']].astype(str).str.replace(r'\.png$', '', regex=True)
df['study_id'] = df['stem'].map(derive_study)
n_study = df['study_id'].nunique()
print('studi unik   :', n_study, '(kartu dataset: 10.643)')
print('pasien unik  :', df[COL['patient']].nunique(), '(kartu dataset: 6.091)')
print('citra        :', len(df))
print('rerata citra per studi :', round(len(df) / n_study, 2))

studi unik   : 10699 (kartu dataset: 10.643)
pasien unik  : 6091 (kartu dataset: 6.091)
citra        : 20327
rerata citra per studi : 1.9


## 5. Membangun flag exclusion

Delapan kategori dibuang dari himpunan pretraining. Dua di antaranya mudah terlewat:

- **Klasifikasi AO yang tidak kosong.** Sebuah citra bisa memuat klasifikasi AO tanpa kotak fraktur, yaitu ketika frakturnya diketahui ada tetapi tidak tervisualisasi jelas pada proyeksi itu. Tanpa filter ini, fraktur okulta masuk ke himpunan bersih dan model belajar bahwa tampilan itu normal.
- **`pronatorsign` dan `softtissue`.** Keduanya tanda tidak langsung cedera, bukan temuan tulang. Citra dengan tanda ini berisiko memuat fraktur okulta.

`text` sengaja tidak difilter. Penanda huruf muncul di hampir seluruh citra, jadi memfilternya menyisakan nol. Model akan mempelajarinya sebagai bagian dari normal, dan itulah alasan tambahan skor triase memakai kuantil ke-95 alih-alih maksimum.

In [8]:
def has(cls):
    return df['stem'].map(lambda s: presence[cls].get(s, 0) > 0)

df['lbl_fracture']     = has('fracture')
df['lbl_metal']        = has('metal')
df['lbl_periosteal']   = has('periostealreaction')
df['lbl_pronator']     = has('pronatorsign')
df['lbl_softtissue']   = has('softtissue')
df['lbl_boneanomaly']  = has('boneanomaly')
df['lbl_bonelesion']   = has('bonelesion')
df['lbl_foreignbody']  = has('foreignbody')
df['n_fracture_box']   = df['stem'].map(lambda s: presence['fracture'].get(s, 0))

ao = df[COL['ao']].astype(str).str.strip().str.lower()
df['tag_ao'] = ~ao.isin(['', 'nan', 'none', '0', 'na'])

def as_bool(col):
    if col is None:
        return pd.Series(False, index=df.index)
    s = df[col]
    if s.dtype == object:
        return s.astype(str).str.strip().str.lower().isin(['1','true','yes','y'])
    return s.fillna(0).astype(float) > 0

df['tag_cast']     = as_bool(COL['cast'])
df['tag_frac_vis'] = as_bool(COL['frac_vis'])

EXCL_STRICT = ['lbl_fracture','tag_frac_vis','tag_ao','lbl_metal','tag_cast',
               'lbl_periosteal','lbl_pronator','lbl_softtissue',
               'lbl_boneanomaly','lbl_bonelesion','lbl_foreignbody']
EXCL_LOOSE  = [c for c in EXCL_STRICT if c != 'tag_cast']

df['clean_strict'] = ~df[EXCL_STRICT].any(axis=1)
df['clean_loose']  = ~df[EXCL_LOOSE].any(axis=1)

print('kontribusi tiap filter (jumlah citra yang dikenai):')
for c in EXCL_STRICT:
    print(f'  {c:16s} {int(df[c].sum()):>6d}')

print()
print('citra bersih KETAT  :', int(df.clean_strict.sum()),
      'dari', int(df[df.clean_strict][COL['patient']].nunique()), 'pasien')
print('citra bersih LONGGAR:', int(df.clean_loose.sum()),
      'dari', int(df[df.clean_loose][COL['patient']].nunique()), 'pasien')
print('gips pada', round(100 * df.tag_cast.mean(), 1), '% citra')

only_ao = df.tag_ao & ~df.lbl_fracture & ~df.tag_frac_vis
print('\nfraktur okulta yang tertangkap khusus oleh filter AO:', int(only_ao.sum()))

kontribusi tiap filter (jumlah citra yang dikenai):
  lbl_fracture      13550
  tag_frac_vis      13550
  tag_ao            14158
  lbl_metal           707
  tag_cast           5776
  lbl_periosteal     2235
  lbl_pronator        566
  lbl_softtissue      439
  lbl_boneanomaly     192
  lbl_bonelesion       42
  lbl_foreignbody       8

citra bersih KETAT  : 5639 dari 2671 pasien
citra bersih LONGGAR: 5672 dari 2682 pasien
gips pada 28.4 % citra

fraktur okulta yang tertangkap khusus oleh filter AO: 773


## 6. Split per pasien

GroupKFold pada `patient_id`, dijalankan sekali di sini dan dipakai seluruh eksperimen. Dengan rata-rata 3,3 citra per pasien, pembagian acak per citra menempatkan pasien yang sama di train dan test.

Fold 0 menjadi test, fold 1 menjadi validasi, sisanya train. Statistik normalisasi dan pemilihan lambda nanti hanya menyentuh fold validasi.

In [9]:
gkf = GroupKFold(n_splits=N_FOLDS)
df['fold'] = -1
for i, (_, idx) in enumerate(gkf.split(df, groups=df[COL['patient']])):
    df.loc[df.index[idx], 'fold'] = i

df['split'] = np.select(
    [df.fold == 0, df.fold == 1],
    ['test', 'val'],
    default='train')

chk = df.groupby('split').agg(
    citra=('stem', 'size'),
    pasien=(COL['patient'], 'nunique'),
    studi=('study_id', 'nunique'),
    bersih_ketat=('clean_strict', 'sum'),
    fraktur=('lbl_fracture', 'sum'))
display(chk)

overlap = (df.groupby(COL['patient'])['split'].nunique() > 1).sum()
print('pasien yang muncul di lebih dari satu split:', overlap, '(harus 0)')
assert overlap == 0

,citra,pasien,studi,bersih_ketat,fraktur
split,,,,,
test,4066,1218,2139,1157,2667
train,12195,3654,6395,3382,8141
val,4066,1219,2165,1100,2742


pasien yang muncul di lebih dari satu split: 0 (harus 0)


## 7. Sebaran usia per band

Statistik normalisasi dihitung per band usia, jadi band yang tipis menghasilkan estimasi yang tidak stabil. Sel ini menandai band dengan kurang dari 50 citra bersih di fold validasi. Ujung bawah rentang hampir pasti tipis, dan kamu perlu tahu itu sekarang, bukan saat menulis Bagian 5.

In [10]:
df['age'] = pd.to_numeric(df[COL['age']], errors='coerce')
print('rentang usia:', round(df.age.min(), 2), 'sampai', round(df.age.max(), 2))
print('usia hilang :', int(df.age.isna().sum()))

df['age_band'] = np.floor(df.age).astype('Int64')
band = (df[df.clean_strict]
        .groupby(['age_band', 'split'])
        .size().unstack(fill_value=0)
        .reindex(columns=['train','val','test'], fill_value=0))
band['total'] = band.sum(axis=1)
display(band)

thin = band[band['val'] < 50].index.tolist()
print('band dengan < 50 citra bersih di fold val:', thin)
print('band ini perlu digabung atau dilebarkan sebelum menghitung (mu_a, sigma_a)')

rentang usia: 0.2 sampai 19.0
usia hilang : 0


split,train,val,test,total
age_band,,,,
0,9,0,0,9
1,46,12,12,70
2,53,16,20,89
3,53,10,17,80
4,62,23,10,95
5,37,25,22,84
6,61,10,21,92
7,93,37,27,157
8,156,48,55,259


band dengan < 50 citra bersih di fold val: [0, 1, 2, 3, 4, 5, 6, 7, 8, 18, 19]
band ini perlu digabung atau dilebarkan sebelum menghitung (mu_a, sigma_a)


## 8. Prapemrosesan

Untuk tiap citra: resize sisi panjang ke 384 dengan mempertahankan rasio aspek, potong intensitas pada persentil 1 dan 99 yang dihitung sebelum padding, skalakan, lalu pad nol simetris ke kanvas 384x384.

Padding dihitung sebelum normalisasi akan mencemari persentilnya dengan piksel nol, jadi urutannya penting. Offset padding disimpan di manifest karena dua hal membutuhkannya: pemetaan kotak fraktur ke ruang 384, dan masking patch padding saat training.

In [11]:
from multiprocessing import Pool, cpu_count

MAXV = 65535 if BIT_DEPTH == 16 else 255
DTYPE = np.uint16 if BIT_DEPTH == 16 else np.uint8

def preprocess_one(args):
    stem, src = args
    try:
        im = cv2.imread(str(src), cv2.IMREAD_UNCHANGED)
        if im is None:
            return (stem, None)
        if im.ndim == 3:
            im = im[..., 0]
        h0, w0 = im.shape
        scale = IMG_SIZE / max(h0, w0)
        nw, nh = max(1, int(round(w0 * scale))), max(1, int(round(h0 * scale)))
        im = cv2.resize(im, (nw, nh), interpolation=cv2.INTER_AREA).astype(np.float32)

        lo, hi = np.percentile(im, [CLIP_LO, CLIP_HI])
        if hi <= lo:
            lo, hi = float(im.min()), float(max(im.max(), im.min() + 1))
        im = np.clip((im - lo) / (hi - lo), 0, 1)

        canvas = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
        px, py = (IMG_SIZE - nw) // 2, (IMG_SIZE - nh) // 2
        canvas[py:py+nh, px:px+nw] = im
        cv2.imwrite(str(OUT / 'images_384' / f'{stem}.png'),
                    (canvas * MAXV).astype(DTYPE))
        return (stem, dict(orig_w=w0, orig_h=h0, scale=scale,
                           new_w=nw, new_h=nh, pad_x=px, pad_y=py,
                           clip_lo=float(lo), clip_hi=float(hi)))
    except Exception as e:
        return (stem, None)

jobs = [(s, img_index[s]) for s in df['stem'] if s in img_index]
print('memproses', len(jobs), 'citra dengan', cpu_count(), 'proses')

geo = {}
with Pool(cpu_count()) as pool:
    for i, (stem, info) in enumerate(pool.imap_unordered(preprocess_one, jobs, chunksize=64)):
        if info is not None:
            geo[stem] = info
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(jobs)}')

print('berhasil:', len(geo), 'gagal:', len(jobs) - len(geo))
for k in ['scale','new_w','new_h','pad_x','pad_y']:
    df[k] = df['stem'].map(lambda s: geo.get(s, {}).get(k, np.nan))
df['preprocessed'] = df['stem'].isin(geo)

memproses 20327 citra dengan 4 proses
  2000/20327
  4000/20327
  6000/20327
  8000/20327
  10000/20327
  12000/20327
  14000/20327
  16000/20327
  18000/20327
  20000/20327
berhasil: 20327 gagal: 0


## 9. Memetakan kotak fraktur ke ruang 384

Label YOLO memakai koordinat ternormalisasi terhadap citra asli. Setelah resize dan padding, koordinatnya bergeser. Sel ini menuliskannya ulang dalam piksel ruang 384 sekaligus dalam indeks patch, sehingga aturan coverage minimal 50% untuk label patch bisa dihitung langsung tanpa transformasi lagi.

In [12]:
rows = []
for stem, boxes in frac_boxes_raw.items():
    g = geo.get(stem)
    if g is None:
        continue
    for (xc, yc, bw, bh) in boxes:
        cx = xc * g['new_w'] + g['pad_x']
        cy = yc * g['new_h'] + g['pad_y']
        w  = bw * g['new_w']
        h  = bh * g['new_h']
        x0, y0 = max(0.0, cx - w/2), max(0.0, cy - h/2)
        x1, y1 = min(float(IMG_SIZE), cx + w/2), min(float(IMG_SIZE), cy + h/2)
        rows.append(dict(stem=stem, x0=x0, y0=y0, x1=x1, y1=y1,
                         patch_i0=int(x0 // PATCH), patch_j0=int(y0 // PATCH),
                         patch_i1=int((x1 - 1e-6) // PATCH),
                         patch_j1=int((y1 - 1e-6) // PATCH)))

fb = pd.DataFrame(rows)
fb.to_csv(OUT / 'fracture_boxes.csv', index=False)
print('kotak fraktur diekspor:', len(fb), 'pada', fb.stem.nunique(), 'citra')
display(fb.head())

area = ((fb.x1 - fb.x0) * (fb.y1 - fb.y0)) / (PATCH * PATCH)
print('\nluas kotak dalam satuan patch 16x16:')
print(area.describe().round(1))

kotak fraktur diekspor: 18090 pada 13550 citra


,stem,x0,y0,x1,y1,patch_i0,patch_j0,patch_i1,patch_j1
0,5533_1055512321_02_WRI-L1_F008,189.074145,268.548864,236.721315,324.182784,11,16,14,20
1,5533_1055512321_02_WRI-L1_F008,143.935005,250.562112,173.191935,275.659968,8,15,10,17
2,2007_0850762755_05_WRI-R2_F008,159.053082,198.309888,215.469078,234.816768,9,12,13,14
3,0313_0947475228_06_WRI-R2_F015,176.518490,189.061248,240.074070,238.694016,11,11,15,14
4,3863_1014196661_02_WRI-L1_M015,176.901020,263.746560,230.300340,299.896320,11,16,14,18



luas kotak dalam satuan patch 16x16:
count    18090.0
mean         6.2
std          4.2
min          0.5
25%          3.4
50%          5.3
75%          7.9
max         61.6
dtype: float64


## 10. Sampel untuk anotasi lempeng pertumbuhan

Seratus citra normal, diambil hanya dari pasien fold validasi supaya anotasinya tidak menyentuh test. Sepuluh di antaranya ditandai untuk dianotasi ketiga anggota tim, sehingga kamu bisa melaporkan kesepakatan antar-anotator di Bagian 4.

Diekspor sebagai PNG 8-bit karena hanya untuk dilihat mata manusia, dan koordinatnya sudah berada di ruang 384 sehingga bisa langsung dipakai sebagai kotak patch.

In [13]:
pool_val = df[(df.split == 'val') & df.clean_strict & df.preprocessed].copy()
pool_val = pool_val.dropna(subset=['age'])

# stratifikasi per band usia supaya tidak semua sampel berasal dari kelompok yang sama
picked = (pool_val.sample(frac=1.0, random_state=SEED)
          .groupby('age_band', group_keys=False)
          .head(max(1, N_ANNOT // max(1, pool_val.age_band.nunique()))))
if len(picked) < N_ANNOT:
    extra = pool_val[~pool_val.stem.isin(picked.stem)].sample(
        N_ANNOT - len(picked), random_state=SEED)
    picked = pd.concat([picked, extra])
picked = picked.head(N_ANNOT).reset_index(drop=True)
picked['overlap'] = picked.index < N_ANNOT_OVERLAP

for s in picked.stem:
    im = cv2.imread(str(OUT / 'images_384' / f'{s}.png'), cv2.IMREAD_UNCHANGED)
    if BIT_DEPTH == 16:
        im = (im / 257).astype(np.uint8)
    cv2.imwrite(str(OUT / 'annot_sample' / f'{s}.png'), im)

picked[['stem', COL['patient'], 'study_id', 'age', 'overlap']].to_csv(
    OUT / 'annot_sample' / 'annot_list.csv', index=False)

template = pd.DataFrame(columns=['stem','struktur','x0','y0','x1','y1','status','annotator'])
template.to_csv(OUT / 'annot_sample' / 'physis_boxes_TEMPLATE.csv', index=False)

print('sampel anotasi:', len(picked), '| overlap tiga anotator:', int(picked.overlap.sum()))
print('sebaran band usia:'); print(picked.age_band.value_counts().sort_index())

sampel anotasi: 100 | overlap tiga anotator: 10
sebaran band usia:
age_band
1     5
2     5
3     5
4     5
5     6
6     5
7     5
8     5
9     5
10    6
11    5
12    8
13    6
14    7
15    5
16    6
17    7
18    4
Name: count, dtype: Int64


## 11. Menulis manifest dan ringkasan

In [14]:
keep = ['stem', COL['patient'], 'study_id', 'age', 'age_band', COL['gender']]
for c in [COL['projection'], COL['laterality']]:
    if c: keep.append(c)
keep += ['fold','split','clean_strict','clean_loose','n_fracture_box',
         'lbl_fracture','lbl_metal','lbl_periosteal','lbl_pronator','lbl_softtissue',
         'lbl_boneanomaly','lbl_bonelesion','lbl_foreignbody',
         'tag_ao','tag_cast','tag_frac_vis',
         'orig_ok','scale','new_w','new_h','pad_x','pad_y','preprocessed']
df['orig_ok'] = df['preprocessed']
manifest = df[[c for c in keep if c in df.columns]].rename(
    columns={COL['patient']: 'patient_id', COL['gender']: 'gender'})
manifest.to_csv(OUT / 'manifest.csv', index=False)

summary = dict(
    n_images=int(len(df)), n_patients=int(df[COL['patient']].nunique()),
    n_studies=int(df.study_id.nunique()),
    n_clean_strict=int(df.clean_strict.sum()), n_clean_loose=int(df.clean_loose.sum()),
    n_clean_strict_train=int(df[(df.split=='train') & df.clean_strict].shape[0]),
    n_clean_strict_val=int(df[(df.split=='val') & df.clean_strict].shape[0]),
    cast_rate=float(df.tag_cast.mean()),
    occult_caught_by_ao=int(only_ao.sum()),
    img_size=IMG_SIZE, patch=PATCH, bit_depth=BIT_DEPTH,
    clip=[CLIP_LO, CLIP_HI], n_folds=N_FOLDS, seed=SEED,
    class_order=CLASSES, thin_val_bands=[int(b) for b in thin],
    preprocess_failed=int((~df.preprocessed).sum()))
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
band.to_csv(OUT / 'age_band_counts.csv')

print(json.dumps(summary, indent=2))

{
  "n_images": 20327,
  "n_patients": 6091,
  "n_studies": 10699,
  "n_clean_strict": 5639,
  "n_clean_loose": 5672,
  "n_clean_strict_train": 3382,
  "n_clean_strict_val": 1100,
  "cast_rate": 0.28415408077925913,
  "occult_caught_by_ao": 773,
  "img_size": 384,
  "patch": 16,
  "bit_depth": 16,
  "clip": [
    1.0,
    99.0
  ],
  "n_folds": 5,
  "seed": 1337,
  "class_order": [
    "boneanomaly",
    "bonelesion",
    "foreignbody",
    "fracture",
    "metal",
    "periostealreaction",
    "pronatorsign",
    "softtissue",
    "text"
  ],
  "thin_val_bands": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    18,
    19
  ],
  "preprocess_failed": 0
}


In [15]:
readme = f'''# Physis — data siap pakai

Dihasilkan oleh notebook Tahap 0 dari GRAZPEDWRI-DX (Kaggle: jasonroggy/grazpedwri-dx, CC0).

## Isi
- manifest.csv           satu baris per citra; kolom `split` dan `fold` sudah final
- fracture_boxes.csv     kotak fraktur di ruang {IMG_SIZE}x{IMG_SIZE} dan indeks patch
- age_band_counts.csv    jumlah citra bersih per band usia per split
- summary.json           seluruh parameter dan angka ringkas
- images_384/            {len(geo)} citra, {IMG_SIZE}x{IMG_SIZE}, {BIT_DEPTH}-bit, sudah dipad
- annot_sample/          {len(picked)} citra normal fold val untuk anotasi lempeng pertumbuhan

## Prapemrosesan
resize sisi panjang ke {IMG_SIZE} (rasio aspek dipertahankan)
-> potong persentil {CLIP_LO}/{CLIP_HI} pada citra sebelum padding
-> skala ke [0,1] -> pad nol simetris ke {IMG_SIZE}x{IMG_SIZE}

Patch yang seluruhnya padding HARUS dikeluarkan dari sampling context, sampling
target, dan peta surprise. Gunakan pad_x, pad_y, new_w, new_h dari manifest:
patch (i,j) valid bila 16*i >= pad_x, 16*(i+1) <= pad_x+new_w, dan setara untuk j.

## Aturan split
GroupKFold {N_FOLDS} fold pada patient_id, seed {SEED}.
fold 0 = test, fold 1 = val, fold 2..4 = train. Tidak ada pasien lintas split.

## Definisi citra bersih
clean_strict membuang: fracture, fracture_visible, klasifikasi AO tidak kosong,
metal, cast, periostealreaction, pronatorsign, softtissue, boneanomaly,
bonelesion, foreignbody.
clean_loose sama tetapi mempertahankan cast.
`text` sengaja tidak difilter karena muncul di hampir seluruh citra.
'''
(OUT / 'README.md').write_text(readme)
print(readme)

# Physis — data siap pakai

Dihasilkan oleh notebook Tahap 0 dari GRAZPEDWRI-DX (Kaggle: jasonroggy/grazpedwri-dx, CC0).

## Isi
- manifest.csv           satu baris per citra; kolom `split` dan `fold` sudah final
- fracture_boxes.csv     kotak fraktur di ruang 384x384 dan indeks patch
- age_band_counts.csv    jumlah citra bersih per band usia per split
- summary.json           seluruh parameter dan angka ringkas
- images_384/            20327 citra, 384x384, 16-bit, sudah dipad
- annot_sample/          100 citra normal fold val untuk anotasi lempeng pertumbuhan

## Prapemrosesan
resize sisi panjang ke 384 (rasio aspek dipertahankan)
-> potong persentil 1.0/99.0 pada citra sebelum padding
-> skala ke [0,1] -> pad nol simetris ke 384x384

Patch yang seluruhnya padding HARUS dikeluarkan dari sampling context, sampling
target, dan peta surprise. Gunakan pad_x, pad_y, new_w, new_h dari manifest:
patch (i,j) valid bila 16*i >= pad_x, 16*(i+1) <= pad_x+new_w, dan setara untuk j.

## Aturan sp

## 12. Zip

Metadata dipisah dari citra supaya berkas kecil bisa diunduh lebih dulu. Citra dipecah menjadi beberapa shard karena unduhan berkas tunggal berukuran besar dari Kaggle sering putus.

Kalau `KEEP_PNG_DIR = True`, direktori PNG dipertahankan dan kamu bisa menyimpan output notebook ini sebagai Kaggle Dataset, lalu meng-attach-nya ke notebook lain tanpa mengunduh apa pun.

In [16]:
ZIPDIR = Path('/kaggle/working/zips'); ZIPDIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIPDIR / 'physis_meta.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['manifest.csv','fracture_boxes.csv','age_band_counts.csv',
              'summary.json','README.md']:
        z.write(OUT / f, f)
    for f in sorted((OUT / 'annot_sample').iterdir()):
        z.write(f, f'annot_sample/{f.name}')
print('physis_meta.zip :', round((ZIPDIR/'physis_meta.zip').stat().st_size/1e6, 1), 'MB')

pngs = sorted((OUT / 'images_384').iterdir())
limit = ZIP_SHARD_GB * 1e9
shard, cur, size = 1, [], 0
def flush(shard, files):
    if not files: return
    p = ZIPDIR / f'physis_images_{shard:02d}.zip'
    with zipfile.ZipFile(p, 'w', zipfile.ZIP_STORED) as z:
        for f in files:
            z.write(f, f'images_384/{f.name}')
    print(p.name, ':', round(p.stat().st_size/1e9, 2), 'GB', f'({len(files)} berkas)')

for f in pngs:
    s = f.stat().st_size
    if size + s > limit and cur:
        flush(shard, cur); shard += 1; cur, size = [], 0
    cur.append(f); size += s
flush(shard, cur)

if not KEEP_PNG_DIR:
    shutil.rmtree(OUT / 'images_384')
    print('\ndirektori PNG dihapus, hanya zip yang tersisa')
print('\nsiap diunduh dari /kaggle/working/zips')

physis_meta.zip : 6.1 MB
physis_images_01.zip : 1.4 GB (9904 berkas)
physis_images_02.zip : 1.4 GB (9926 berkas)
physis_images_03.zip : 0.07 GB (497 berkas)

direktori PNG dihapus, hanya zip yang tersisa

siap diunduh dari /kaggle/working/zips


## Langkah berikutnya

1. Unduh `physis_meta.zip` lebih dulu dan baca `summary.json`. Angka `n_clean_strict` menentukan apakah Tahap A bisa dilatih dari awal atau harus diinisialisasi dari ImageNet, dan apakah E2 muat dalam anggaran komputasi. Kalau `n_clean_strict` jauh di bawah 3.000, pakai `clean_loose` dan catat keputusan gips itu di Bagian 6 paper.
2. Periksa `thin_val_bands` di `summary.json`. Band yang tercantum di situ perlu digabung sebelum menghitung statistik normalisasi.
3. Unduh shard citra, atau simpan output notebook sebagai Kaggle Dataset lalu tarik lewat Kaggle API di mesin vast.ai.
4. Bagikan `annot_sample/` ke tim untuk anotasi lempeng pertumbuhan, hasilnya diisikan ke `physis_boxes_TEMPLATE.csv`.
5. Commit `manifest.csv`, `fracture_boxes.csv`, `summary.json`, dan notebook ini ke repo. Split sudah final sejak titik ini, dan seluruh eksperimen membacanya dari berkas yang sama.